# Notebook 43 - KLT and Kalman downstream gates

Notebook 42 showed that the corrected TimTrack mask/doHough path can pass the MATLAB raw-export gate. This notebook splits the remaining work into two downstream gates:

1. **UltraTrack/KLT gate**: compare the current Python KLT-derived fascicle geometry against MATLAB's pre-Kalman geometry copied into `Region.Fascicle.fas_x_original` / `fas_y_original`.
2. **Kalman gate**: compare the current Python experimental geometric Kalman output against MATLAB's final smoothed `Region` outputs, while keeping the MATLAB 2-state Kalman contract separate from the current Python 4-state filter.

Expected status before porting: TimTrack gate is ready; KLT and MATLAB 2-state Kalman are not ready yet.

In [1]:
from pathlib import Path
import json
import math
import sys

import numpy as np
import pandas as pd
from scipy.io import loadmat

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ultrasound_tracker.geometry import line_angles_batch, line_lengths_batch, normalize_angle
from ultrasound_tracker.matlab_compat import metric_row

OUT = ROOT / "results" / "notebook43_klt_kalman_downstream_gates"
OUT.mkdir(parents=True, exist_ok=True)

MATLAB_EXPORT = Path("/Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat")
NB42_METRICS = ROOT / "results" / "notebook42_timtrack_full_pipeline_gate" / "parity_metrics.csv"
PY_KLT = ROOT / "results" / "klt_sequence_features_arrays.npz"
PY_KALMAN = ROOT / "results" / "ultratimtrack_geometric_kalman_features_arrays.npz"

for path in [MATLAB_EXPORT, NB42_METRICS, PY_KLT, PY_KALMAN]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Output folder:", OUT)
print("MATLAB export:", MATLAB_EXPORT)
print("Python KLT:", PY_KLT)
print("Python experimental Kalman:", PY_KALMAN)

Output folder: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook43_klt_kalman_downstream_gates
MATLAB export: /Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat
Python KLT: /Users/grosbedou/PycharmProjects/NDORMS/results/klt_sequence_features_arrays.npz
Python experimental Kalman: /Users/grosbedou/PycharmProjects/NDORMS/results/ultratimtrack_geometric_kalman_features_arrays.npz


## Load MATLAB and Python artifacts

The MATLAB numeric export stores fascicle point pairs as MATLAB cell-like arrays. In MATLAB convention for these fields, the first point is the deep attachment and the second point is the superficial attachment. For parity with the Python segment convention, this notebook converts them to `[x_sup, y_sup, x_deep, y_deep]`.

Important references from `UltraTimTrack.m`:

- KLT feature detection uses `detectMinEigenFeatures(..., 'FilterSize', 11, 'MinQuality', 0.005)` and selects the strongest 300 fascicle points for Hough ROI tracking.
- MATLAB point tracking uses `vision.PointTracker('NumPyramidLevels', 4, 'MaxIterations', 50, 'MaxBidirectionalError', inf, 'BlockSize', handles.BlockSize)`.
- Fascicle and aponeurosis motion are propagated with separate affine transforms from `estimateGeometricTransform2D(..., 'affine', 'MaxDistance', 50)`.
- MATLAB saves pre-Kalman fascicle geometry by copying `fas_x` / `fas_y` into `fas_x_original` / `fas_y_original` immediately after affine propagation.
- The MATLAB smoother is a 2-state model over `[superficial attachment x, fascicle alpha]`, with `Q = handles.Q * dx^2`, followed by an RTS-style backward pass.

In [2]:
mat_root = loadmat(MATLAB_EXPORT, simplify_cells=True)["UTT_numeric_export"]
region = mat_root["Region"]
fascicle = region["Fascicle"]

mm_per_px = float(mat_root["ID"]) / int(mat_root["vidHeight"])
mat_n = int(mat_root["NumFrames"])
video_shape = (int(mat_root["vidHeight"]), int(mat_root["vidWidth"]))

py_klt = np.load(PY_KLT, allow_pickle=True)
py_kalman = np.load(PY_KALMAN, allow_pickle=True)

print({
    "matlab_frames": mat_n,
    "video_shape": video_shape,
    "mm_per_px": mm_per_px,
    "python_klt_frames": int(len(py_klt["frame"])),
    "python_kalman_frames": int(len(py_kalman["frame"])),
})

print("MATLAB Fascicle fields:", sorted(fascicle.keys()))
print("Python KLT keys:", sorted(py_klt.files))
print("Python Kalman keys:", sorted(py_kalman.files))

{'matlab_frames': 2666, 'video_shape': (562, 706), 'mm_per_px': 0.09021352313167261, 'python_klt_frames': 2667, 'python_kalman_frames': 2667}
MATLAB Fascicle fields: ['A', 'K', 'X_minus', 'X_plus', 'alpha', 'fas_p', 'fas_p_minus', 'fas_x', 'fas_x_end', 'fas_x_original', 'fas_y', 'fas_y_end', 'fas_y_original']
Python KLT keys: ['deep_apo_angle_deg', 'deep_apo_lines', 'deep_attachments', 'fascicle_angle_deg', 'fascicle_length_px', 'fascicle_lines', 'fascicle_segments', 'frame', 'klt_affine_used', 'klt_dx', 'klt_dy', 'klt_n_points', 'pennation_angle_deg', 'success', 'sup_apo_lines', 'sup_attachments', 'time_s']
Python Kalman keys: ['frame', 'klt_length_px', 'klt_pennation_angle_deg', 'klt_segments', 'success', 'time_s', 'timtrack_length_px', 'timtrack_pennation_angle_deg', 'timtrack_segments', 'used_klt_prediction', 'used_timtrack_measurement', 'utt_deep_apo_angle_deg', 'utt_deep_attachments', 'utt_fascicle_angle_deg', 'utt_fascicle_length_px', 'utt_gain_diag', 'utt_innovation_norm', 'utt

In [3]:
def matlab_pair_series_to_array(values: object, expected_n: int | None = None) -> np.ndarray:
    """Return an array shaped (n_frames, 2) from MATLAB cell/vector fascicle fields."""
    out = []
    for value in np.asarray(values, dtype=object).reshape(-1):
        arr = np.asarray(value, dtype=float).reshape(-1)
        if arr.size < 2:
            out.append([np.nan, np.nan])
        else:
            out.append(arr[:2])
    result = np.asarray(out, dtype=float)
    if expected_n is not None:
        result = result[:expected_n]
    return result


def matlab_fascicle_segments(x_field: str, y_field: str, expected_n: int | None = None) -> np.ndarray:
    """Convert MATLAB [deep; superficial] fascicle endpoints to Python [sup, deep] segments."""
    x = matlab_pair_series_to_array(fascicle[x_field], expected_n=expected_n)
    y = matlab_pair_series_to_array(fascicle[y_field], expected_n=expected_n)
    return np.column_stack([x[:, 1], y[:, 1], x[:, 0], y[:, 0]])


def matlab_region_array(name: str, expected_n: int | None = None) -> np.ndarray:
    arr = np.asarray(region[name], dtype=float).reshape(-1)
    if expected_n is not None:
        arr = arr[:expected_n]
    return arr


def align_by_frame(npz, key: str, frame_numbers: np.ndarray) -> np.ndarray:
    frames = np.asarray(npz["frame"], dtype=int)
    values = np.asarray(npz[key])
    frame_to_index = {int(frame): idx for idx, frame in enumerate(frames)}
    missing = [int(frame) for frame in frame_numbers if int(frame) not in frame_to_index]
    if missing:
        raise KeyError(f"Missing {len(missing)} frames for {key}: first missing={missing[:10]}")
    idx = np.asarray([frame_to_index[int(frame)] for frame in frame_numbers], dtype=int)
    return values[idx]


def normalized_segment_angles_deg(segments: np.ndarray) -> np.ndarray:
    raw = line_angles_batch(np.asarray(segments, dtype=float), degrees=True)
    return np.asarray([normalize_angle(value, degrees=True) for value in raw], dtype=float)


def metric(label: str, ref, est, unit: str, target_rmse: float | None = None, target_mae: float | None = None) -> dict:
    row = metric_row(label, ref, est)
    row["unit"] = unit
    row["target_rmse"] = target_rmse
    row["target_mae"] = target_mae
    row["passes"] = bool(
        (target_rmse is None or row["rmse"] <= target_rmse)
        and (target_mae is None or row["mae"] <= target_mae)
    )
    return row


def save_metrics(rows: list[dict], path: Path) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    ordered = [
        "comparison", "unit", "n", "bias", "mae", "rmse", "corr",
        "target_mae", "target_rmse", "passes",
    ]
    df = df[[col for col in ordered if col in df.columns]]
    df.to_csv(path, index=False)
    return df

## TimTrack gate hand-off

Notebook 42 is the upstream gate. Here we read its metrics and keep only the corrected TimTrack rows for the hand-off decision. The final FL/PEN/ANG rows in notebook 42 are intentionally excluded here because those are downstream KLT/Kalman outputs, not the corrected mask/doHough TimTrack gate.

In [4]:
nb42 = pd.read_csv(NB42_METRICS)
nb42["gate"] = np.where(nb42["comparison"].str.startswith("timtrack_"), "timtrack", "downstream_final")

# The length tolerance is the user's 2 mm target converted into pixels.
timtrack_thresholds = {
    "timtrack_alpha_deg": {"mae": 0.5, "rmse": 1.0},
    "timtrack_phi_vs_python_pen_deg": {"mae": 0.5, "rmse": 1.0},
    "timtrack_formula_faslen_px": {"mae": 2.0 / mm_per_px, "rmse": 2.0 / mm_per_px},
    "timtrack_gamma_deep_apo_deg": {"mae": 0.5, "rmse": 1.0},
    "timtrack_betha_super_apo_deg": {"mae": 0.5, "rmse": 1.0},
    "timtrack_super_pos_y1": {"mae": 2.0, "rmse": 2.0},
    "timtrack_super_pos_y2": {"mae": 2.0, "rmse": 2.0},
    "timtrack_deep_pos_y1": {"mae": 2.0, "rmse": 2.0},
    "timtrack_deep_pos_y2": {"mae": 2.0, "rmse": 2.0},
}

timtrack_rows = nb42[nb42["gate"] == "timtrack"].copy()
timtrack_rows["target_mae"] = timtrack_rows["comparison"].map(lambda name: timtrack_thresholds[name]["mae"])
timtrack_rows["target_rmse"] = timtrack_rows["comparison"].map(lambda name: timtrack_thresholds[name]["rmse"])
timtrack_rows["passes"] = (timtrack_rows["mae"] <= timtrack_rows["target_mae"]) & (timtrack_rows["rmse"] <= timtrack_rows["target_rmse"])

timtrack_out = OUT / "timtrack_gate_handoff_from_nb42.csv"
timtrack_rows.to_csv(timtrack_out, index=False)

print("Saved:", timtrack_out)
display(timtrack_rows[["comparison", "n", "mae", "rmse", "target_mae", "target_rmse", "passes"]])
print("TimTrack hand-off passes:", bool(timtrack_rows["passes"].all()))

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook43_klt_kalman_downstream_gates/timtrack_gate_handoff_from_nb42.csv


,comparison,n,mae,rmse,target_mae,target_rmse,passes
3,timtrack_alpha_deg,9,0.222222,0.408248,0.500000,1.000000,True
4,timtrack_phi_vs_python_pen_deg,9,0.229521,0.393471,0.500000,1.000000,True
5,timtrack_formula_faslen_px,9,11.347822,18.476314,22.169625,22.169625,True
6,timtrack_gamma_deep_apo_deg,9,0.082311,0.137994,0.500000,1.000000,True
7,timtrack_betha_super_apo_deg,9,0.050026,0.083137,0.500000,1.000000,True
8,timtrack_super_pos_y1,9,1.274274,1.444055,2.000000,2.000000,True
9,timtrack_super_pos_y2,9,0.991992,1.042095,2.000000,2.000000,True
10,timtrack_deep_pos_y1,9,0.683884,1.110706,2.000000,2.000000,True
11,timtrack_deep_pos_y2,9,0.463099,0.625233,2.000000,2.000000,True


TimTrack hand-off passes: True


## UltraTrack/KLT gate

This compares the current Python KLT artifact against MATLAB `fas_x_original` / `fas_y_original`, which is the geometry copied before MATLAB's state estimator and smoother change `fas_x` / `fas_y`.

Because the current Python KLT implementation is not an UltraTrack parity port, this gate is expected to fail. The purpose is to freeze the reference variable, coordinate convention, and metric outputs for the upcoming KLT port.

In [5]:
frame_numbers = np.arange(mat_n, dtype=int)

mat_klt_segments = matlab_fascicle_segments("fas_x_original", "fas_y_original", expected_n=mat_n)

# Existing Python artifacts are 0-based image coordinates; MATLAB export is 1-based.
py_klt_segments = align_by_frame(py_klt, "fascicle_segments", frame_numbers).astype(float) + 1.0

mat_klt_angle = normalized_segment_angles_deg(mat_klt_segments)
py_klt_angle = normalized_segment_angles_deg(py_klt_segments)
mat_klt_length = line_lengths_batch(mat_klt_segments)
py_klt_length = line_lengths_batch(py_klt_segments)

endpoint_target_px = 2.0
length_target_px = 2.0 / mm_per_px
angle_target_deg = 1.0

klt_metrics = save_metrics([
    metric("klt_x_sup_px", mat_klt_segments[:, 0], py_klt_segments[:, 0], "px", target_rmse=endpoint_target_px),
    metric("klt_y_sup_px", mat_klt_segments[:, 1], py_klt_segments[:, 1], "px", target_rmse=endpoint_target_px),
    metric("klt_x_deep_px", mat_klt_segments[:, 2], py_klt_segments[:, 2], "px", target_rmse=endpoint_target_px),
    metric("klt_y_deep_px", mat_klt_segments[:, 3], py_klt_segments[:, 3], "px", target_rmse=endpoint_target_px),
    metric("klt_angle_deg", mat_klt_angle, py_klt_angle, "deg", target_rmse=angle_target_deg),
    metric("klt_length_px", mat_klt_length, py_klt_length, "px", target_rmse=length_target_px),
    metric("klt_length_mm", mat_klt_length * mm_per_px, py_klt_length * mm_per_px, "mm", target_rmse=2.0),
], OUT / "klt_gate_metrics.csv")

print("Saved:", OUT / "klt_gate_metrics.csv")
display(klt_metrics)
print("KLT gate passes:", bool(klt_metrics["passes"].all()))

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook43_klt_kalman_downstream_gates/klt_gate_metrics.csv


,comparison,unit,n,bias,mae,rmse,corr,target_mae,target_rmse,passes
0,klt_x_sup_px,px,2666,81.906519,87.280049,106.764971,-0.829786,None,2.000000,False
1,klt_y_sup_px,px,2666,13.727965,15.905644,18.813061,-0.052020,None,2.000000,False
2,klt_x_deep_px,px,2666,-104.939328,104.939328,115.082864,0.985783,None,2.000000,False
3,klt_y_deep_px,px,2666,-25.404230,25.497664,28.791218,-0.124202,None,2.000000,False
4,klt_angle_deg,deg,2666,-8.978846,8.978868,10.823260,-0.870375,None,1.000000,False
5,klt_length_px,px,2666,159.266467,159.266467,186.941159,0.499383,None,22.169625,False
6,klt_length_mm,mm,2666,14.367989,14.367989,16.864621,0.499383,None,2.000000,False


KLT gate passes: False


In [6]:
klt_gap_rows = [
    {
        "area": "Feature detector",
        "matlab_contract": "detectMinEigenFeatures(FilterSize=11, MinQuality=0.005)",
        "current_python": "cv2.goodFeaturesToTrack(maxCorners=50, qualityLevel=0.3, minDistance=7, blockSize=7)",
        "status": "mismatch",
    },
    {
        "area": "Fascicle feature count",
        "matlab_contract": "selectStrongest(300) for Hough ROI",
        "current_python": "maxCorners defaults to 50",
        "status": "mismatch",
    },
    {
        "area": "Point tracker",
        "matlab_contract": "vision.PointTracker NumPyramidLevels=4, MaxIterations=50, MaxBidirectionalError=inf, BlockSize=[21, 71]",
        "current_python": "cv2.calcOpticalFlowPyrLK winSize=(15,15), maxLevel=3, criteria=(30, 0.01)",
        "status": "mismatch",
    },
    {
        "area": "Geometry propagation",
        "matlab_contract": "estimateGeometricTransform2D(..., affine, MaxDistance=50); transform fascicle endpoints",
        "current_python": "existing artifact stores already-derived segments; no MATLAB affine parity checkpoint yet",
        "status": "missing parity checkpoint",
    },
    {
        "area": "Aponeurosis propagation",
        "matlab_contract": "separate aponeurosis affine transform and ROI propagation",
        "current_python": "not yet validated against MATLAB awarp/sup/deep propagation",
        "status": "missing parity checkpoint",
    },
    {
        "area": "Reinitialization",
        "matlab_contract": "re-detect fascicle points below 100; re-detect aponeurosis points below 500",
        "current_python": "re-detects only if fewer than 10 tracked points in simple KLT wrapper",
        "status": "mismatch",
    },
]

klt_gap = pd.DataFrame(klt_gap_rows)
klt_gap_path = OUT / "klt_implementation_gap_inventory.csv"
klt_gap.to_csv(klt_gap_path, index=False)
print("Saved:", klt_gap_path)
display(klt_gap)

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook43_klt_kalman_downstream_gates/klt_implementation_gap_inventory.csv


,area,matlab_contract,current_python,status
0,Feature detector,"detectMinEigenFeatures(FilterSize=11, MinQuali...","cv2.goodFeaturesToTrack(maxCorners=50, quality...",mismatch
1,Fascicle feature count,selectStrongest(300) for Hough ROI,maxCorners defaults to 50,mismatch
2,Point tracker,"vision.PointTracker NumPyramidLevels=4, MaxIte...","cv2.calcOpticalFlowPyrLK winSize=(15,15), maxL...",mismatch
3,Geometry propagation,"estimateGeometricTransform2D(..., affine, MaxD...",existing artifact stores already-derived segme...,missing parity checkpoint
4,Aponeurosis propagation,separate aponeurosis affine transform and ROI ...,not yet validated against MATLAB awarp/sup/dee...,missing parity checkpoint
5,Reinitialization,re-detect fascicle points below 100; re-detect...,re-detects only if fewer than 10 tracked point...,mismatch


## Kalman gate

This is intentionally **not** a MATLAB 2-state Kalman implementation yet. It compares the existing Python experimental 4-state fusion artifact to MATLAB's final smoothed output so we have a baseline and a clean fail condition.

The parity implementation should be added separately, with the MATLAB state contract preserved:

- State: `[fas_x(2), alpha]`, i.e. superficial fascicle attachment x and fascicle alpha.
- Process noise: `Q = Q_parameter * dx^2`.
- Forward estimator arrays: `X_plus`, `X_minus`, `fas_p`, `fas_p_minus`.
- Backward smoother: RTS-style pass using `A = Pcorr / Ppred` and reconstructing fascicle geometry from smoothed x and alpha.

The current `ultrasound_tracker.ultratimtrack_kalman` module is a 4-state model `[x_sup, y_sup, angle, length]` and should remain separate while the MATLAB 2-state model is ported.

In [7]:
mat_final_segments = matlab_fascicle_segments("fas_x", "fas_y", expected_n=mat_n)
py_kalman_segments = align_by_frame(py_kalman, "utt_segments", frame_numbers).astype(float) + 1.0

mat_final_angle_from_segments = normalized_segment_angles_deg(mat_final_segments)
py_kalman_angle_from_segments = normalized_segment_angles_deg(py_kalman_segments)
mat_final_length_from_segments_mm = line_lengths_batch(mat_final_segments) * mm_per_px
py_kalman_length_from_segments_mm = line_lengths_batch(py_kalman_segments) * mm_per_px

mat_region_fl_mm = matlab_region_array("fas_length", expected_n=mat_n)
mat_region_pen_deg = matlab_region_array("fas_pen", expected_n=mat_n)
mat_region_ang_deg = matlab_region_array("fas_ang", expected_n=mat_n)

py_kalman_ang_deg = align_by_frame(py_kalman, "utt_fascicle_angle_deg", frame_numbers).astype(float)
py_kalman_pen_deg = align_by_frame(py_kalman, "utt_pennation_angle_deg", frame_numbers).astype(float)
py_kalman_fl_mm = align_by_frame(py_kalman, "utt_fascicle_length_px", frame_numbers).astype(float) * mm_per_px

kalman_metrics = save_metrics([
    metric("kalman_ANG_deg", mat_region_ang_deg, py_kalman_ang_deg, "deg", target_rmse=1.0),
    metric("kalman_PEN_deg", mat_region_pen_deg, py_kalman_pen_deg, "deg", target_rmse=1.0),
    metric("kalman_FL_mm", mat_region_fl_mm, py_kalman_fl_mm, "mm", target_rmse=2.0),
    metric("kalman_segment_angle_deg", mat_final_angle_from_segments, py_kalman_angle_from_segments, "deg", target_rmse=1.0),
    metric("kalman_segment_length_mm", mat_final_length_from_segments_mm, py_kalman_length_from_segments_mm, "mm", target_rmse=2.0),
], OUT / "kalman_gate_metrics.csv")

print("Saved:", OUT / "kalman_gate_metrics.csv")
display(kalman_metrics)
print("Kalman gate passes:", bool(kalman_metrics["passes"].all()))

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook43_klt_kalman_downstream_gates/kalman_gate_metrics.csv


,comparison,unit,n,bias,mae,rmse,corr,target_mae,target_rmse,passes
0,kalman_ANG_deg,deg,2666,-8.975239,8.975418,10.689862,-0.767142,None,1.0,False
1,kalman_PEN_deg,deg,2666,-8.577530,8.577645,10.154128,-0.662508,None,1.0,False
2,kalman_FL_mm,mm,2666,17.137766,17.138140,19.086755,0.570227,None,2.0,False
3,kalman_segment_angle_deg,deg,2666,-8.975239,8.975418,10.689862,-0.767142,None,1.0,False
4,kalman_segment_length_mm,mm,2666,17.540351,17.540745,19.574314,0.568633,None,2.0,False


Kalman gate passes: False


In [8]:
kalman_contract_fields = ["X_plus", "X_minus", "fas_p", "fas_p_minus", "A", "K", "alpha", "fas_x", "fas_y", "fas_x_original", "fas_y_original"]
rows = []
for field in kalman_contract_fields:
    value = fascicle.get(field)
    arr = np.asarray(value, dtype=object) if value is not None else np.asarray([])
    rows.append({
        "field": field,
        "present_in_matlab_export": value is not None,
        "shape": str(arr.shape),
        "first_value_preview": str(arr.reshape(-1)[0])[:120] if arr.size else "",
    })

kalman_contract = pd.DataFrame(rows)
kalman_contract_path = OUT / "matlab_2state_kalman_contract_fields.csv"
kalman_contract.to_csv(kalman_contract_path, index=False)
print("Saved:", kalman_contract_path)
display(kalman_contract)

print("MATLAB Q parameter:", mat_root.get("Q"))

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook43_klt_kalman_downstream_gates/matlab_2state_kalman_contract_fields.csv


,field,present_in_matlab_export,shape,first_value_preview
0,X_plus,True,"(2666,)",[733. 18.52048864]
1,X_minus,True,"(2666,)",[733. 18.52048864]
2,fas_p,True,"(2666,)",[0 1]
3,fas_p_minus,True,"(2666,)",[0 0]
4,A,True,"(2665,)",1
5,K,True,"(2666,)",0.0
6,alpha,True,"(2666,)",18.52048863720337
7,fas_x,True,"(2666,)",[-27. 733.]
8,fas_y,True,"(2666,)",[309.01343161 54.41875512]
9,fas_x_original,True,"(2666,)",[-27 733]


MATLAB Q parameter: 0.01


## Readiness decision

This notebook keeps the gates separate:

- **TimTrack gate**: ready for hand-off from notebook 42.
- **KLT gate**: not ready; next work is an UltraTrack parity port with MATLAB-like feature detection, LK settings, affine propagation, aponeurosis propagation, and pre-Kalman geometry exports.
- **Kalman gate**: not ready; next work is a separate MATLAB 2-state Kalman + RTS smoother implementation. The existing 4-state fusion remains an experimental branch and should not be used as the MATLAB parity target.

In [9]:
readiness_rows = [
    {
        "gate": "TimTrack mask/doHough",
        "reference": "NB36 raw MATLAB export + NB42 parity metrics",
        "artifact": str(NB42_METRICS.relative_to(ROOT)),
        "passes": bool(timtrack_rows["passes"].all()),
        "next_action": "Use as upstream measurement input for downstream gates.",
    },
    {
        "gate": "UltraTrack/KLT",
        "reference": "MATLAB Region.Fascicle.fas_x_original / fas_y_original",
        "artifact": str((OUT / "klt_gate_metrics.csv").relative_to(ROOT)),
        "passes": bool(klt_metrics["passes"].all()),
        "next_action": "Port MATLAB-like KLT and affine propagation; save pre-Kalman geometry for validation.",
    },
    {
        "gate": "MATLAB 2-state Kalman",
        "reference": "MATLAB Region final outputs and Fascicle X_plus/X_minus/fas_p/fas_p_minus/A",
        "artifact": str((OUT / "kalman_gate_metrics.csv").relative_to(ROOT)),
        "passes": bool(kalman_metrics["passes"].all()),
        "next_action": "Implement separate 2-state model [superficial x, alpha] plus RTS smoother; keep current 4-state filter separate.",
    },
]

readiness = pd.DataFrame(readiness_rows)
readiness_path = OUT / "downstream_gate_readiness.csv"
readiness.to_csv(readiness_path, index=False)

print("Saved:", readiness_path)
display(readiness)

summary = {
    "timtrack_gate_passes": bool(timtrack_rows["passes"].all()),
    "klt_gate_passes": bool(klt_metrics["passes"].all()),
    "kalman_gate_passes": bool(kalman_metrics["passes"].all()),
    "klt_angle_rmse_deg": float(klt_metrics.loc[klt_metrics["comparison"] == "klt_angle_deg", "rmse"].iloc[0]),
    "klt_length_rmse_px": float(klt_metrics.loc[klt_metrics["comparison"] == "klt_length_px", "rmse"].iloc[0]),
    "kalman_ang_rmse_deg": float(kalman_metrics.loc[kalman_metrics["comparison"] == "kalman_ANG_deg", "rmse"].iloc[0]),
    "kalman_pen_rmse_deg": float(kalman_metrics.loc[kalman_metrics["comparison"] == "kalman_PEN_deg", "rmse"].iloc[0]),
    "kalman_fl_rmse_mm": float(kalman_metrics.loc[kalman_metrics["comparison"] == "kalman_FL_mm", "rmse"].iloc[0]),
}
summary_path = OUT / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Saved:", summary_path)
print(json.dumps(summary, indent=2))

Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook43_klt_kalman_downstream_gates/downstream_gate_readiness.csv


,gate,reference,artifact,passes,next_action
0,TimTrack mask/doHough,NB36 raw MATLAB export + NB42 parity metrics,results/notebook42_timtrack_full_pipeline_gate...,True,Use as upstream measurement input for downstre...
1,UltraTrack/KLT,MATLAB Region.Fascicle.fas_x_original / fas_y_...,results/notebook43_klt_kalman_downstream_gates...,False,Port MATLAB-like KLT and affine propagation; s...
2,MATLAB 2-state Kalman,MATLAB Region final outputs and Fascicle X_plu...,results/notebook43_klt_kalman_downstream_gates...,False,Implement separate 2-state model [superficial ...


Saved: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook43_klt_kalman_downstream_gates/summary.json
{
  "timtrack_gate_passes": true,
  "klt_gate_passes": false,
  "kalman_gate_passes": false,
  "klt_angle_rmse_deg": 10.823259709648635,
  "klt_length_rmse_px": 186.9411591466729,
  "kalman_ang_rmse_deg": 10.689861554721636,
  "kalman_pen_rmse_deg": 10.154128214886308,
  "kalman_fl_rmse_mm": 19.086755187688738
}
